## Импорты

In [ ]:
import asyncio
import csv
import json
import random
import re
import time
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from typing import Dict, List, Optional, Tuple
from urllib.parse import urljoin, urlparse
import httpx
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
!pip install undetected_chromedriver

In [ ]:
import undetected_chromedriver as uc

Константы и вспомогательные функции

In [ ]:
BASE = "https://hh.ru"

RU_MONTHS = {
    "января": 1, "февраля": 2, "марта": 3, "апреля": 4, "мая": 5, "июня": 6,
    "июля": 7, "августа": 8, "сентября": 9, "октября": 10, "ноября": 11, "декабря": 12,
    "январь": 1, "февраль": 2, "март": 3, "апрель": 4, "май": 5, "июнь": 6,
    "июль": 7, "август": 8, "сентябрь": 9, "октябрь": 10, "ноябрь": 11, "декабрь": 12,
}

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def soup_text(el) -> str:
    return clean_text(el.get_text(" ", strip=True)) if el else ""

def sleep_soft(a=0.9, b=1.8):
    time.sleep(random.uniform(a, b))

def normalize_resume_url(href: str) -> str:
    full = urljoin(BASE, href)
    p = urlparse(full)
    return f"{p.scheme}://{p.netloc}{p.path}"

def parse_ru_relative_date(raw: str, base: date) -> Optional[date]:
    if not raw:
        return None
    t = raw.lower()
    if "сегодня" in t:
        return base
    if "вчера" in t:
        return base - timedelta(days=1)
    m = re.search(r"(\d{1,2})\s+([а-яё]+)\s+(\d{4})", t)
    if m:
        d = int(m.group(1))
        mon = RU_MONTHS.get(m.group(2))
        y = int(m.group(3))
        if mon:
            try:
                return date(y, mon, d)
            except ValueError:
                return None
    return None

def extract_label_value(soup: BeautifulSoup, label_prefix: str) -> str:
    lp = label_prefix.lower()
    for p in soup.select("p"):
        txt = soup_text(p)
        if txt.lower().startswith(lp):
            parts = txt.split(":", 1)
            return clean_text(parts[1]) if len(parts) == 2 else txt
    return ""


Модель данных резюме

In [ ]:
@dataclass
class ResumeItem:
    url: str
    desired_position: str = ""
    exp_industries: str = ""
    resume_update_year: str = ""
    age: str = ""
    birth_date: str = ""
    gender: str = ""
    education_level: str = ""
    university: str = ""
    faculty: str = ""
    graduation_year: str = ""
    specialty: str = ""
    work_schedule: str = ""
    last_job_duration: str = ""
    last_company: str = ""
    last_job_title: str = ""
    last_job_duties: str = ""
    prev_company: str = ""
    prev_job_duties: str = ""
    prev_job_duration: str = ""
    location: str = ""
    skills: List[str] = field(default_factory=list)
    about_me: str = ""
    desired_salary: str = ""
    resume_update_date_raw: str = ""
    resume_update_date: str = ""
    status: str = ""
    archive_date: str = ""
    online_status: str = ""
    total_experience: str = ""

Сбор ссылок с выдачи по резюме

In [ ]:
def build_driver(headless: bool = False) -> uc.Chrome:
    options = uc.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1400,1000")
    driver = uc.Chrome(options=options)
    driver.set_page_load_timeout(60)
    return driver

def wait_dom(driver, timeout=25):
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )

def wait_serp_loaded(driver, timeout=25):
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, '[data-qa="resume-serp__resume"]'))
    )

def get_page_soup(driver) -> BeautifulSoup:
    return BeautifulSoup(driver.page_source, "lxml")

def parse_serp_cards_for_links_and_meta(soup: BeautifulSoup) -> Tuple[List[str], Dict[str, Dict[str, str]], Optional[str]]:
    cards = soup.select('[data-qa="resume-serp__resume"]')
    if not cards:
        return [], {}, None

    first_hash = cards[0].get("data-resume-hash") or None

    links: List[str] = []
    meta: Dict[str, Dict[str, str]] = {}
    seen_local = set()

    for c in cards:
        status = ""
        status_node = c.select_one('[data-qa="resume-card-tags"] [class*="magritte-tag__label"]')
        if status_node:
            status = soup_text(status_node)

        update_raw = ""
        for sp in c.select("span"):
            t = soup_text(sp)
            if t.lower().startswith("обновлено"):
                update_raw = t
                break

        a = c.select_one('a[data-qa="serp-item__title"]')
        if a and a.get("href"):
            u = normalize_resume_url(a["href"])
            if u not in seen_local:
                seen_local.add(u)
                links.append(u)
                meta[u] = {"status": status, "update_raw": update_raw}
            continue

        rh = c.get("data-resume-hash")
        if rh:
            u = f"{BASE}/resume/{rh}"
            if u not in seen_local:
                seen_local.add(u)
                links.append(u)
                meta[u] = {"status": status, "update_raw": update_raw}

    return links, meta, first_hash

def collect_all_serp_with_selenium(
    driver,
    search_url: str,
    max_pages: Optional[int] = None,
    max_resumes: Optional[int] = None,
) -> Tuple[List[str], Dict[str, Dict[str, str]], Dict[str, str], str]:
    driver.get(search_url)
    wait_dom(driver)
    wait_serp_loaded(driver)
    sleep_soft()

    ua = driver.execute_script("return navigator.userAgent;")
    cookies_list = driver.get_cookies()
    cookies = {c["name"]: c["value"] for c in cookies_list if c.get("domain") and "hh.ru" in c.get("domain")}

    all_urls: List[str] = []
    all_meta: Dict[str, Dict[str, str]] = {}
    seen = set()

    pages_done = 0
    prev_first_hash = None

    while True:
        if max_pages is not None and pages_done >= max_pages:
            break

        soup = get_page_soup(driver)
        urls, meta, first_hash = parse_serp_cards_for_links_and_meta(soup)

        for u in urls:
            if u not in seen:
                seen.add(u)
                all_urls.append(u)
                all_meta[u] = meta.get(u, {})

                if max_resumes is not None and len(all_urls) >= max_resumes:
                    break
        if max_resumes is not None and len(all_urls) >= max_resumes:
            break

        pages_done += 1

        next_btns = driver.find_elements(By.CSS_SELECTOR, '[data-qa="pager-next"]')
        if not next_btns:
            break

        try:
            if prev_first_hash is not None and first_hash == prev_first_hash:
                break
            prev_first_hash = first_hash

            pager = driver.find_elements(By.CSS_SELECTOR, '[data-qa="pager-block"]')
            if pager:
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", pager[0])
                sleep_soft(0.3, 0.8)

            next_btns[0].click()

            WebDriverWait(driver, 25).until(
                lambda d: (
                    BeautifulSoup(d.page_source, "lxml")
                    .select_one('[data-qa="resume-serp__resume"]')
                    and (
                        BeautifulSoup(d.page_source, "lxml")
                        .select_one('[data-qa="resume-serp__resume"]')
                        .get("data-resume-hash", "") != (first_hash or "")
                    )
                )
            )
            sleep_soft()
        except Exception:
            break

    cookies_list = driver.get_cookies()
    cookies = {c["name"]: c["value"] for c in cookies_list if c.get("domain") and "hh.ru" in c.get("domain")}

    return all_urls, all_meta, cookies, ua

Парсинг отдельной страницы резюме

In [ ]:
def parse_education(soup: BeautifulSoup) -> Dict[str, str]:
    edu = {"level": "", "university": "", "faculty": "", "specialty": "", "graduation_year": ""}
    block = soup.select_one('[data-qa="resume-block-education"]')
    if not block:
        return edu

    h2 = block.select_one("h2")
    if h2:
        edu["level"] = soup_text(h2)

    year_node = block.select_one(".bloko-column_l-2")
    if year_node:
        y = re.search(r"\b(19|20)\d{2}\b", soup_text(year_node))
        if y:
            edu["graduation_year"] = y.group(0)

    edu["university"] = soup_text(block.select_one('[data-qa="resume-block-education-name"]'))

    org = block.select_one('[data-qa="resume-block-education-organization"]')
    if org:
        spans = [soup_text(x) for x in org.select("span") if soup_text(x)]
        if len(spans) >= 1:
            edu["faculty"] = spans[0]
        if len(spans) >= 2:
            edu["specialty"] = spans[1]
        elif len(spans) == 1:
            edu["specialty"] = spans[0]

    return edu

def parse_skills(soup: BeautifulSoup) -> List[str]:
    skills = []
    for tag in soup.select(
        '[data-qa="skills-table"] .label--_PWdo2ImNdEM1M0A span, '
        '[data-qa="skills-table"] .bloko-tag__section_text span'
    ):
        t = soup_text(tag)
        if t:
            skills.append(t)
    out, seen = [], set()
    for s in skills:
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out

def parse_experience_blocks(soup: BeautifulSoup) -> List[Dict[str, str]]:
    block = soup.select_one('[data-qa="resume-block-experience"]')
    if not block:
        return []

    items = []
    pos_nodes = block.select('[data-qa="resume-block-experience-position"]')

    for pos_node in pos_nodes:
        root = pos_node
        for _ in range(10):
            if not root:
                break
            if root.find(attrs={"data-qa": "resume-block-experience-description"}):
                break
            root = root.parent
        if not root:
            continue

        company = ""
        strongs = root.select(".bloko-text_strong span")
        if strongs:
            company = soup_text(strongs[0])

        location = soup_text(root.select_one("p"))

        industries = ""
        ind = root.select_one(".resume-block__experience-industries p span")
        if ind:
            industries = soup_text(ind)

        position = soup_text(pos_node)

        desc = ""
        desc_node = root.select_one('[data-qa="resume-block-experience-description"]')
        if desc_node:
            for br in desc_node.find_all("br"):
                br.replace_with("\n")
            desc = desc_node.get_text("\n", strip=True).strip()

        duration = soup_text(root.select_one(".bloko-text_tertiary"))

        items.append({
            "company": company,
            "location": location,
            "industries": industries,
            "position": position,
            "description": desc,
            "duration": duration
        })

    return items

def parse_resume(html: str, url: str, meta: Dict[str, str]) -> ResumeItem:
    soup = BeautifulSoup(html, "lxml")
    item = ResumeItem(url=url)

    item.desired_position = soup_text(soup.select_one('[data-qa="resume-block-title-position"]'))
    item.desired_salary = soup_text(soup.select_one('[data-qa="resume-block-salary"]'))

    item.gender = soup_text(soup.select_one('[data-qa="resume-personal-gender"]'))
    item.age = soup_text(soup.select_one('[data-qa="resume-personal-age"]'))

    bd_raw = soup_text(soup.select_one('[data-qa="resume-personal-birthday"]'))
    if bd_raw:
        d = parse_ru_relative_date(bd_raw, base=date.today())
        item.birth_date = d.isoformat() if d else bd_raw

    item.location = soup_text(soup.select_one('[data-qa="resume-personal-address"]'))
    item.online_status = soup_text(soup.select_one(".resume-online-status"))

    item.skills = parse_skills(soup)

    about = soup.select_one('[data-qa="resume-block-skills-content"]')
    if about:
        for br in about.find_all("br"):
            br.replace_with("\n")
        item.about_me = clean_text(about.get_text("\n", strip=True))

    edu = parse_education(soup)
    item.education_level = edu["level"]
    item.university = edu["university"]
    item.faculty = edu["faculty"]
    item.specialty = edu["specialty"]
    item.graduation_year = edu["graduation_year"]

    item.work_schedule = (
        extract_label_value(soup, "График работы")
        or extract_label_value(soup, "Тип занятости")
    )

    exp_title = soup.select_one('[data-qa="resume-block-experience"] .resume-block__title-text')
    item.total_experience = soup_text(exp_title)

    exps = parse_experience_blocks(soup)
    if exps:
        last = exps[0]
        item.last_company = last.get("company", "")
        item.last_job_title = last.get("position", "")
        item.last_job_duties = last.get("description", "")
        item.last_job_duration = last.get("duration", "")
        item.exp_industries = last.get("industries", "")

    if len(exps) >= 2:
        prev = exps[1]
        item.prev_company = prev.get("company", "")
        item.prev_job_duties = prev.get("description", "")
        item.prev_job_duration = prev.get("duration", "")

    item.status = meta.get("status", "")
    item.resume_update_date_raw = meta.get("update_raw", "")
    if item.resume_update_date_raw:
        parsed = parse_ru_relative_date(item.resume_update_date_raw, base=date.today())
        if parsed:
            item.resume_update_date = parsed.isoformat()
            item.resume_update_year = str(parsed.year)

    for sp in soup.select("span"):
        txt = soup_text(sp)
        if "архив" in txt.lower():
            item.archive_date = txt
            break

    return item

Асинхронная функция резюме

In [ ]:
async def fetch_text_async(client: httpx.AsyncClient, url: str, sem: asyncio.Semaphore, retries=3) -> str:
    async with sem:
        for attempt in range(retries):
            try:
                await asyncio.sleep(random.uniform(0.10, 0.35))
                r = await client.get(url, follow_redirects=True)
                r.raise_for_status()
                return r.text
            except Exception:
                if attempt == retries - 1:
                    raise
                await asyncio.sleep(random.uniform(0.8, 1.8))
    return ""

async def fetch_and_parse_resumes(
    resume_urls: List[str],
    link_meta: Dict[str, Dict[str, str]],
    cookies: Dict[str, str],
    ua: str,
    concurrency: int = 6,
    max_resumes: Optional[int] = None,
) -> List[ResumeItem]:
    if max_resumes is not None:
        resume_urls = resume_urls[:max_resumes]

    headers = {
        "User-Agent": ua,
        "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Connection": "keep-alive",
        "Referer": BASE,
    }

    sem = asyncio.Semaphore(concurrency)

    async with httpx.AsyncClient(headers=headers, cookies=cookies, timeout=45.0) as client:
        async def one(u: str) -> Optional[ResumeItem]:
            try:
                html = await fetch_text_async(client, u, sem)
                return parse_resume(html, u, link_meta.get(u, {}))
            except Exception:
                return None

        tasks = [one(u) for u in resume_urls]
        res = await asyncio.gather(*tasks)
        return [x for x in res if x is not None]

Сохранение результата в CSV и JSON

In [ ]:
def save_results(results: List[ResumeItem], out_csv: str, out_json: str):
    if not results:
        print("No results.")
        return

    with open(out_csv, "w", newline="", encoding="utf-8") as f